# ============================================================
# [Data Integration] Dental Offices in Berlin
# ============================================================

# Install dependencies (if not already installed)
# %pip install osmnx geopandas pandas

# ============================================================
# 0. Imports
# ============================================================

In [559]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
from pathlib import Path
import numpy as np


# ============================================================
# 1. Data Extraction & Initial Inspection
# ============================================================


In [560]:
# 1.1 OSM Settings
ox.settings.use_cache = True
ox.settings.log_console = True

# 1.2 Fetch Dental Offices from OpenStreetMap
tags = {"amenity": "dentist"}

dental_offices_osm = ox.features.features_from_place(
    "Berlin, Germany",
    tags=tags
)

print(f"Number of dental office entries fetched: {len(dental_offices_osm)}")
print(dental_offices_osm.head(3).T.to_string())
print(dental_offices_osm['healthcare:speciality'].value_counts())

Number of dental office entries fetched: 798
element                                                  node                                                                                                                                                   
id                                                  304183504                                                                       313539258                                                          325161442
geometry                         POINT (13.612096 52.5114112)                                                   POINT (13.3553052 52.5488382)                                      POINT (13.1804772 52.5088434)
addr:city                                              Berlin                                                                          Berlin                                                                NaN
addr:country                                               DE                                                          

/Users/alex/anaconda3/envs/ds/lib/python3.10/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/Users/alex/anaconda3/envs/ds/lib/python3.10/site-packages/shapely/set_operations.py:451: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)


In [561]:
# Save raw data for reproducibility
dental_offices_osm.to_csv("../sources/raw_osm_dental_offices_v_01_19_2026.csv", index=False)
dental_offices_osm.to_file("../sources/raw_osm_dental_offices_v_01_19_2026.geojson", driver="GeoJSON")

# Inspect dataset
dental_offices_osm.info()
print(dental_offices_osm.columns.tolist())

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Columns: 104 entries, geometry to type
dtypes: geometry(1), object(103)
memory usage: 689.8+ KB
['geometry', 'addr:city', 'addr:country', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'amenity', 'health_facility:type', 'health_specialty:dentistry', 'healthcare', 'medical_system:western', 'office', 'operator', 'description', 'level', 'name', 'opening_hours', 'wheelchair', 'phone', 'toilets:wheelchair', 'website', 'contact:website', 'contact:email', 'contact:fax', 'contact:phone', 'check_date:opening_hours', 'check_date', 'email', 'fax', 'healthcare:speciality', 'air_conditioning', 'internet_access', 'internet_access:fee', 'payment:bank_transfer', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'payment:paypal', 'source', 'toilets', 'entrance', 'opening_hours:signed', 'wheelchair:description', 'emergency', 'name:de', 'name:en', 'payment:cont

# ============================================================
# 2. Data Selection & Column Standardization
# ============================================================

In [ ]:
# Select key columns relevant for dental officesс
columns = [
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "level",
    "opening_hours",
    "check_date",
    "healthcare:speciality",
    "wheelchair",
    "wheelchair:description",
    "phone",
    "email",
    "website",
    "geometry",
    "health_facility:type",
    "health_specialty:oral_surgery",
    "health_specialty:orthodontics",
    "health_specialty:periodontology"
]

# Filter the dataset to keep only the selected columns
dental_offices = dental_offices_osm[[c for c in columns if c in dental_offices_osm.columns]].copy()

# ============================================================
# Next Processing Steps (Roadmap for future PRs / scripts)
# ============================================================
# 1. Name normalization
#    - Standardize names such as "Zahnarztpraxis", "Drs.", and other practice naming conventions.
# 2. Address cleaning
#    - Format street names and house numbers to a consistent structure.
# 3. Category mapping
#    - Map specialization fields into a controlled vocabulary for consistency.
# 4. Deduplication logic
#    - Detect and handle overlaps between OSM entries and official city registries.


# ============================================================
# 3. Speciality Mapping
# ============================================================

In [ ]:
# Mapping of OSM health specialty columns to standardized speciality names
# Only consider values marked as "yes" or "main"
special_cols = {
    "health_specialty:oral_surgery": "oral_surgery",
    "health_specialty:orthodontics": "orthodontics",
    "health_specialty:periodontology": "periodontology"
}

def compute_speciality(row):
    """
    Determine standardized speciality for a dental office.
    
    Priority:
    1. Use 'healthcare:speciality' if non-empty
    2. Check boolean-style specialty columns ("yes" or "main")
    3. Default to "Unknown"
    """
    val = row.get("healthcare:speciality")
    if pd.notna(val) and str(val).strip() != "":
        return str(val).strip()
    
    for col, name in special_cols.items():
        cell = row.get(col)
        if pd.notna(cell) and str(cell).lower() in ["yes", "main"]:
            return name
    
    return "Unknown"

# Apply speciality mapping
dental_offices["speciality"] = dental_offices.apply(compute_speciality, axis=1)

# Drop original speciality columns to avoid redundancy
dental_offices.drop(
    columns=["healthcare:speciality"] + list(special_cols.keys()), inplace=True
)

dental_offices.head()

name     addr:street addr:housenumber  \
element id                                                                
node    304183504                  NaN  Hönower Straße               75   
        313539258  Zahnzentrum Wedding    Müllerstraße              34a   
        325161442             A. Nejad             NaN              NaN   
        345236220    Dr. Beate Lengert  Kurfürstendamm              218   
        391394177      Serpil Hartfiel  Kollwitzstraße               77   

                  addr:postcode addr:city level  \
element id                                        
node    304183504         12623    Berlin   NaN   
        313539258         13353    Berlin     2   
        325161442           NaN       NaN   NaN   
        345236220         10719    Berlin   NaN   
        391394177         10435    Berlin   NaN   

                                                       opening_hours  \
element id                                                             
node    304183504                                                NaN   
        313539258  Mo 09:00-19:00; Tu 09:00-18:00; We 09:00-17:00...   
        325161442  Mo-Tu 09:00-19:00; We 09:00-14:00; Th 09:00-19...   
        345236220                                                NaN   
        391394177  Mo,Tu,Th 08:00-19:00; We 18:00-18:00; Fr 08:00...   

                  check_date wheelchair wheelchair:description  \
element id                                                       
node    304183504        NaN        NaN                    NaN   
        313539258        NaN        yes                    NaN   
        325161442        NaN        yes                    NaN   
        345236220        NaN        NaN                    NaN   
        391394177        NaN         no                    NaN   

                              phone email                          website  \
element id                                                                   
node    304183504               NaN   NaN                              NaN   
        313539258               NaN   NaN                              NaN   
        325161442  +49 30 361 91 06   NaN                              NaN   
        345236220               NaN   NaN  http://www.dr-beate-lengert.de/   
        391394177               NaN   NaN                              NaN   

                                    geometry health_facility:type speciality  
element id                                                                    
node    304183504   POINT (13.6121 52.51141)               office    Unknown  
        313539258  POINT (13.35531 52.54884)                  NaN    Unknown  
        325161442  POINT (13.18048 52.50884)                  NaN    Unknown  
        345236220  POINT (13.32814 52.50272)                  NaN    Unknown  
        391394177  POINT (13.41899 52.53755)                  NaN    Unknown

# ============================================================
# 4. Geometry Processing
# ============================================================

In [ ]:

# Ensure point geometry
dental_offices["geometry"] = dental_offices["geometry"].apply(
    lambda g: g if g.geom_type == "Point" else g.representative_point()
)

# Extract latitude and longitude
dental_offices["latitude"] = dental_offices.geometry.y
dental_offices["longitude"] = dental_offices.geometry.x

# Standardize address columns
dental_offices = dental_offices.rename(columns={
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:city": "city"
})
dental_offices.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 798 entries, ('node', 304183504) to ('way', 293129382)
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   name                    767 non-null    object  
 1   street                  582 non-null    object  
 2   housenumber             582 non-null    object  
 3   postcode                535 non-null    object  
 4   city                    527 non-null    object  
 5   level                   90 non-null     object  
 6   opening_hours           602 non-null    object  
 7   check_date              140 non-null    object  
 8   wheelchair              286 non-null    object  
 9   wheelchair:description  8 non-null      object  
 10  phone                   289 non-null    object  
 11  email                   88 non-null     object  
 12  website                 306 non-null    object  
 13  geometry                798 non-null   